In [1]:
# !pip install -r requirements.txt
# !pip install fg-data-profiling

In [1]:

from langgraph.graph import START, END, StateGraph

import os

import pandas as pd
from pydantic import BaseModel, Field, ValidationError, ConfigDict, field_validator
from typing import Optional, TypedDict, List, Annotated, Any, Sequence, Literal

from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv

from data_profiling import ProfileReport


from collections import Counter as C

from langgraph.graph.message import  add_messages

from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, ChatMessage, ToolMessage, AIMessage, SystemMessage, AnyMessage, HumanMessage

from langgraph.checkpoint.memory import MemorySaver

from langgraph.prebuilt import ToolNode

import json

In [3]:
load_dotenv()

AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_NAME="gpt-4.1_1" #.env or this model : same :     working

# print("AZURE_OPENAI_API_VERSION : ", AZURE_OPENAI_API_VERSION)
# print("AZURE_OPENAI_DEPLOYMENT_NAME  : ", AZURE_OPENAI_DEPLOYMENT_NAME)


def setup_llm() -> AzureChatOpenAI:
    return AzureChatOpenAI(
        api_version=AZURE_OPENAI_API_VERSION,
        azure_deployment=AZURE_OPENAI_DEPLOYMENT_NAME,
        temperature=0.2,
    )

llm_model = setup_llm()

# single query
resp = llm_model.invoke("write 2 lines about narendra modi")
print(resp.content)


Narendra Modi is the Prime Minister of India, known for his dynamic leadership and economic reforms. He has played a significant role in shaping India's domestic and international policies.


In [4]:
df = pd.read_csv("patient_data.csv")
df.shape

(100, 18)

In [5]:

profile = ProfileReport(df, title="Patients_Data Profile Report")
profile.to_file("patients_data_profile_report.html")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 18/18 [00:00<00:00, 458.46it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [6]:
df.isna().sum().sum()


np.int64(0)

In [7]:
print(list(df.columns))


['patient_id', 'patient_name', 'age', 'gender', 'diagnosis', 'current_medication', 'lab_test', 'lab_value', 'lab_unit', 'last_visit_date', 'next_scheduled_visit', 'days_since_last_visit', 'missed_last_appointment', 'vitals_bp_systolic', 'vitals_bp_diastolic', 'vitals_heart_rate', 'vitals_spo2', 'notes']


In [8]:
df.head(2)


,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.


In [9]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   patient_id               100 non-null    object 
 1   patient_name             100 non-null    object 
 2   age                      100 non-null    int64  
 3   gender                   100 non-null    object 
 4   diagnosis                100 non-null    object 
 5   current_medication       100 non-null    object 
 6   lab_test                 100 non-null    object 
 7   lab_value                100 non-null    float64
 8   lab_unit                 100 non-null    object 
 9   last_visit_date          100 non-null    object 
 10  next_scheduled_visit     100 non-null    object 
 11  days_since_last_visit    100 non-null    int64  
 12  missed_last_appointment  100 non-null    object 
 13  vitals_bp_systolic       100 non-null    int64  
 14  vitals_bp_diastolic      10

In [10]:

# Data loading and validation

REQUIRED_COLUMNS = [
    "patient_id",
    "patient_name",
    "age",
    "gender",
    "diagnosis",
    "current_medication",
    "lab_test",
    "lab_value",
    "lab_unit",
    "last_visit_date",
    "next_scheduled_visit",
    "days_since_last_visit",
    "missed_last_appointment",
    "vitals_bp_systolic",
    "vitals_bp_diastolic",
    "vitals_heart_rate",
    "vitals_spo2",
    "notes",
]

NUMERIC_COLUMNS = [
    "age",
    "lab_value",
    "days_since_last_visit",
    "vitals_bp_systolic",
    "vitals_bp_diastolic",
    "vitals_heart_rate",
    "vitals_spo2",
]

DATE_COLUMNS = [
    "last_visit_date",
    "next_scheduled_visit",
]

In [11]:
missing_columns = sorted(set(REQUIRED_COLUMNS) - set(df.columns))
if missing_columns:
    raise ValueError(
        f"CSV is missing required columns: {missing_columns}"
    )

if df["patient_id"].isna().any():
    raise ValueError("patient_id cannot contain null values.")

df["patient_id"] = (
    df["patient_id"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [12]:
df.head()

,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.
2,P0003,Rohan Mehta,45,Male,"Type 2 Diabetes, CKD","Insulin Glargine 10U, Losartan 50mg",HbA1c,11.4,%,2024-08-04,2024-10-03,149,No,132,86,64,91,Medication dose adjusted this visit.
3,P0004,Anjali Singh,34,Female,"COPD, Heart Failure","Tiotropium 18mcg, Furosemide 40mg",FEV1%,33.8,%,2024-01-23,2024-04-22,343,No,153,70,93,94,Patient reports no new symptoms.
4,P0005,Vikram Nair,51,Male,"Hypertension, Heart Failure","Amlodipine 5mg, Furosemide 40mg",BP Systolic,124.9,mmHg,2024-04-26,2024-06-25,249,No,134,71,82,94,Patient education provided on diet and lifestyle.


In [13]:
df["patient_id"]

0     P0001
1     P0002
2     P0003
3     P0004
4     P0005
      ...  
95    P0096
96    P0097
97    P0098
98    P0099
99    P0100
Name: patient_id, Length: 100, dtype: object

In [14]:
from collections import Counter as C

In [15]:
C(df["patient_id"])

Counter({'P0001': 1,
         'P0002': 1,
         'P0003': 1,
         'P0004': 1,
         'P0005': 1,
         'P0006': 1,
         'P0007': 1,
         'P0008': 1,
         'P0009': 1,
         'P0010': 1,
         'P0011': 1,
         'P0012': 1,
         'P0013': 1,
         'P0014': 1,
         'P0015': 1,
         'P0016': 1,
         'P0017': 1,
         'P0018': 1,
         'P0019': 1,
         'P0020': 1,
         'P0021': 1,
         'P0022': 1,
         'P0023': 1,
         'P0024': 1,
         'P0025': 1,
         'P0026': 1,
         'P0027': 1,
         'P0028': 1,
         'P0029': 1,
         'P0030': 1,
         'P0031': 1,
         'P0032': 1,
         'P0033': 1,
         'P0034': 1,
         'P0035': 1,
         'P0036': 1,
         'P0037': 1,
         'P0038': 1,
         'P0039': 1,
         'P0040': 1,
         'P0041': 1,
         'P0042': 1,
         'P0043': 1,
         'P0044': 1,
         'P0045': 1,
         'P0046': 1,
         'P0047': 1,
         'P00

In [16]:
if not df["patient_id"].is_unique:
    raise ValueError("patient_id must be unique.")



In [17]:
if not df["patient_id"].str.fullmatch(r"P\d{4,}").all():
    invalid_ids = sorted(
        df.loc[~df["patient_id"].str.fullmatch(r"P\d{4,}"), "patient_id"]
        .unique()
        .tolist()
    )
    raise ValueError(
        f"Invalid patient ID format. Expected values such as P0001. "
        f"Invalid IDs: {invalid_ids}"
    )


In [18]:
NUMERIC_COLUMNS

['age',
 'lab_value',
 'days_since_last_visit',
 'vitals_bp_systolic',
 'vitals_bp_diastolic',
 'vitals_heart_rate',
 'vitals_spo2']

In [19]:
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")

In [20]:
df[NUMERIC_COLUMNS]

,age,lab_value,days_since_last_visit,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2
0,68,9.9,251,174,70,95,96
1,29,317.6,352,174,91,72,97
2,45,11.4,149,132,86,64,91
3,34,33.8,343,153,70,93,94
4,51,124.9,249,134,71,82,94
...,...,...,...,...,...,...,...
95,30,74.8,316,122,89,87,95
96,54,142.0,115,140,67,102,95
97,56,1033.8,177,140,77,65,97
98,70,9.8,340,121,101,71,91


In [21]:
for column in NUMERIC_COLUMNS:
    df[column] = pd.to_numeric(df[column], errors="coerce")




In [22]:
for column in DATE_COLUMNS:
    df[column] = pd.to_datetime(df[column], errors="coerce")



In [23]:
df.head(2)

,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.


In [24]:
if (df["age"].dropna() < 0).any():
    raise ValueError("Age cannot be negative.")

if (df["days_since_last_visit"].dropna() < 0).any():
    raise ValueError("days_since_last_visit cannot be negative.")

In [25]:
df.head(2)

,patient_id,patient_name,age,gender,diagnosis,current_medication,lab_test,lab_value,lab_unit,last_visit_date,next_scheduled_visit,days_since_last_visit,missed_last_appointment,vitals_bp_systolic,vitals_bp_diastolic,vitals_heart_rate,vitals_spo2,notes
0,P0001,Aarav Sharma,68,Male,Type 2 Diabetes,Metformin 500mg,HbA1c,9.9,%,2024-04-24,2024-06-08,251,No,174,70,95,96,Patient reports good medication adherence.
1,P0002,Priya Patel,29,Male,Heart Failure,Furosemide 40mg,BNP,317.6,pg/mL,2024-01-14,2024-02-28,352,No,174,91,72,97,Patient reports no new symptoms.


In [26]:
valid_missed_values = {"yes", "no", ""}


In [27]:
observed_missed_values = set(
    df["missed_last_appointment"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
)


In [28]:

invalid_missed_values = observed_missed_values - valid_missed_values

if invalid_missed_values:
    raise ValueError(
        "missed_last_appointment must contain only Yes, No, or blank values. "
        f"Invalid values: {sorted(invalid_missed_values)}"
    )


In [29]:
def missed_mask() -> pd.Series:
    return (
        df["missed_last_appointment"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .eq("yes")
    )


In [31]:
print("Dataset shape:", df.shape)


Dataset shape: (100, 18)


In [32]:
print("\nMissing values:")
print(df.isna().sum().rename("missing_values"))


Missing values:
patient_id                 0
patient_name               0
age                        0
gender                     0
diagnosis                  0
current_medication         0
lab_test                   0
lab_value                  0
lab_unit                   0
last_visit_date            0
next_scheduled_visit       0
days_since_last_visit      0
missed_last_appointment    0
vitals_bp_systolic         0
vitals_bp_diastolic        0
vitals_heart_rate          0
vitals_spo2                0
notes                      0
Name: missing_values, dtype: int64


In [34]:

print("\nDiagnosis distribution:")
print(
    df["diagnosis"]
    .value_counts(dropna=False)
    .rename("patient_count")
)


Diagnosis distribution:
diagnosis
Heart Failure                    12
Hypothyroidism                    9
Hypertension, Anemia              8
Type 2 Diabetes, Hypertension     7
Osteoarthritis                    7
COPD                              7
Depression                        7
Type 2 Diabetes                   6
Asthma                            6
Anemia                            6
COPD, Heart Failure               6
Hypertension, Heart Failure       5
Chronic Kidney Disease            5
Hypertension                      5
Type 2 Diabetes, CKD              4
Name: patient_count, dtype: int64


In [35]:
print("\nMissed-appointment distribution:")
print(
    df["missed_last_appointment"]
    .value_counts(dropna=False)
    .rename("patient_count")
)


Missed-appointment distribution:
missed_last_appointment
No     85
Yes    15
Name: patient_count, dtype: int64


In [75]:
# Model-facing data minimization
# patient_name is deliberately excluded.


# Note:
# Clinical notes can still contain sensitive information. In a
# production environment, notes should be redacted or filtered
# before being sent to an external model.
MODEL_SAFE_COLUMNS = [
    "patient_id",
    "age",
    "gender",
    "diagnosis",
    "current_medication",
    "lab_test",
    "lab_value",
    "lab_unit",
    "last_visit_date",
    "next_scheduled_visit",
    "days_since_last_visit",
    "missed_last_appointment",
    "vitals_bp_systolic",
    "vitals_bp_diastolic",
    "vitals_heart_rate",
    "vitals_spo2",
    "notes",
]


In [76]:
def model_safe_records(frame: pd.DataFrame) -> str:
    safe_frame = frame.loc[:, MODEL_SAFE_COLUMNS].copy()
    return safe_frame.to_json(
        orient="records",
        date_format="iso",
    )

In [77]:
import re

In [78]:
# Pydantic schemas

class PatientLookup(BaseModel):
    model_config = ConfigDict(extra="forbid")

    patient_id: str = Field(
        ...,
        description="Patient ID, for example P0001",
    )

    @field_validator("patient_id")
    @classmethod
    def validate_patient_id(cls, value: str) -> str:
        normalized = value.strip().upper()

        if not re.fullmatch(r"P\d{4,}", normalized):
            raise ValueError(
                "patient_id must match the format P0001."
            )

        return normalized


In [79]:
# Pydantic schemas

class PatientSearch(BaseModel):
    model_config = ConfigDict(extra="forbid")

    diagnosis: str | None = Field(
        default=None,
        description="Optional diagnosis text",
    )

    lab_test: str | None = Field(
        default=None,
        description="Optional lab test text",
    )

    note_keyword: str | None = Field(
        default=None,
        description="Optional keyword in clinical notes",
    )

    limit: int = Field(
        default=20,
        ge=1,
        le=50,
        description="Maximum number of records",
    )


In [80]:
class PriorityAction(BaseModel):
    model_config = ConfigDict(extra="forbid")

    patient_id: str = Field(
        ...,
        description="Existing patient ID",
    )

    priority: Literal["High", "Medium", "Low"]

    evidence: list[str] = Field(
        ...,
        min_length=1,
    )

    recommended_follow_up: list[str] = Field(
        ...,
        min_length=1,
    )

In [81]:
class AgentReport(BaseModel):
    model_config = ConfigDict(extra="forbid")

    summary: str = Field(..., min_length=1)

    prioritized_actions: list[PriorityAction]

    safety_note: str = Field(..., min_length=1)

    @field_validator("safety_note")
    @classmethod
    def require_clinician_review_language(cls, value: str) -> str:
        lowered = value.lower()

        required_phrases = [
            "clinician review",
            "qualified clinician",
            "medical review",
            "healthcare professional",
        ]

        if not any(phrase in lowered for phrase in required_phrases):
            raise ValueError(
                "safety_note must explicitly require qualified clinician review."
            )

        return value

In [82]:
from threading import Lock

# Audit logging
AUDIT_LOG: list[dict[str, Any]] = []
AUDIT_LOCK = Lock()

In [83]:
from datetime import datetime, timezone

def write_audit_event(
    event_type: str,
    thread_id: str,
    details: dict[str, Any] | None = None,
) -> None:
    """
    Writes metadata only.

    Avoid storing full prompts, clinical notes, medication details,
    or complete tool results in the audit log.
    """
    event = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "event_type": event_type,
        "thread_id": thread_id,
        "details": details or {},
    }

    with AUDIT_LOCK:
        AUDIT_LOG.append(event)

def get_audit_log() -> list[dict[str, Any]]:
    with AUDIT_LOCK:
        return list(AUDIT_LOG)



In [84]:



# Deterministic tools


@tool(args_schema=PatientLookup)
def get_patient(patient_id: str) -> str:
    """
    Get one patient's model-safe clinical snapshot using patient_id.

    patient_name is intentionally excluded from the result.
    """
    normalized_id = patient_id.strip().upper()

    result = df[
        df["patient_id"].astype(str).str.upper() == normalized_id
    ]

    if result.empty:
        return json.dumps({
            "error": f"Patient ID {normalized_id} was not found."
        })

    return model_safe_records(result)


@tool
def get_missed_appointments() -> str:
    """
    Get model-safe records for patients whose last appointment was missed.
    """
    columns = [
        "patient_id",
        "diagnosis",
        "lab_test",
        "lab_value",
        "lab_unit",
        "days_since_last_visit",
        "missed_last_appointment",
        "vitals_bp_systolic",
        "vitals_bp_diastolic",
        "vitals_heart_rate",
        "vitals_spo2",
        "notes",
    ]

    result = df.loc[missed_mask(), columns].sort_values(
        "days_since_last_visit",
        ascending=False,
    )

    return result.to_json(
        orient="records",
        date_format="iso",
    )


@tool(args_schema=PatientSearch)
def search_patients(
    diagnosis: str | None = None,
    lab_test: str | None = None,
    note_keyword: str | None = None,
    limit: int = 20,
) -> str:
    """
    Search patient records using optional filters.

    Only model-safe fields are returned.
    """
    result = df.copy()

    if diagnosis:
        result = result[
            result["diagnosis"]
            .astype(str)
            .str.contains(
                diagnosis,
                case=False,
                na=False,
                regex=False,
            )
        ]

    if lab_test:
        result = result[
            result["lab_test"]
            .astype(str)
            .str.contains(
                lab_test,
                case=False,
                na=False,
                regex=False,
            )
        ]

    if note_keyword:
        result = result[
            result["notes"]
            .astype(str)
            .str.contains(
                note_keyword,
                case=False,
                na=False,
                regex=False,
            )
        ]

    return model_safe_records(result.head(limit))


@tool
def get_cohort_summary() -> str:
    """
    Get aggregate clinic context.

    Raw values from different lab tests or units must not be compared.
    """
    summary = {
        "patient_count": int(len(df)),
        "missed_appointment_count": int(missed_mask().sum()),
        "diagnosis_counts": (
            df["diagnosis"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "lab_test_counts": (
            df["lab_test"]
            .value_counts(dropna=False)
            .to_dict()
        ),
        "days_since_last_visit_summary": (
            df["days_since_last_visit"]
            .describe()
            .round(1)
            .to_dict()
        ),
    }

    return json.dumps(summary, default=str)


In [85]:
TOOLS = [
    get_patient,
    get_missed_appointments,
    search_patients,
    get_cohort_summary,
]

TOOL_MAP = {
    tool_item.name: tool_item
    for tool_item in TOOLS
}

In [86]:
print(TOOLS)

[StructuredTool(name='get_patient', description="Get one patient's model-safe clinical snapshot using patient_id.\n\npatient_name is intentionally excluded from the result.", args_schema=<class '__main__.PatientLookup'>, func=<function get_patient at 0x000002B6A6F95940>), StructuredTool(name='get_missed_appointments', description='Get model-safe records for patients whose last appointment was missed.', args_schema=<class 'langchain_core.utils.pydantic.get_missed_appointments'>, func=<function get_missed_appointments at 0x000002B6A702A8E0>), StructuredTool(name='search_patients', description='Search patient records using optional filters.\n\nOnly model-safe fields are returned.', args_schema=<class '__main__.PatientSearch'>, func=<function search_patients at 0x000002B6A6F94EA0>), StructuredTool(name='get_cohort_summary', description='Get aggregate clinic context.\n\nRaw values from different lab tests or units must not be compared.', args_schema=<class 'langchain_core.utils.pydantic.get

In [87]:
print(TOOL_MAP)

{'get_patient': StructuredTool(name='get_patient', description="Get one patient's model-safe clinical snapshot using patient_id.\n\npatient_name is intentionally excluded from the result.", args_schema=<class '__main__.PatientLookup'>, func=<function get_patient at 0x000002B6A6F95940>), 'get_missed_appointments': StructuredTool(name='get_missed_appointments', description='Get model-safe records for patients whose last appointment was missed.', args_schema=<class 'langchain_core.utils.pydantic.get_missed_appointments'>, func=<function get_missed_appointments at 0x000002B6A702A8E0>), 'search_patients': StructuredTool(name='search_patients', description='Search patient records using optional filters.\n\nOnly model-safe fields are returned.', args_schema=<class '__main__.PatientSearch'>, func=<function search_patients at 0x000002B6A6F94EA0>), 'get_cohort_summary': StructuredTool(name='get_cohort_summary', description='Get aggregate clinic context.\n\nRaw values from different lab tests or 

In [88]:

# Deterministic safety policy checker


# These patterns are intentionally conservative.
# They are not a replacement for a clinical safety system.
BLOCKED_PATTERNS = [
    r"\bprescribe\b",
    r"\bprescription\b",
    r"\bstart\s+(?:a\s+)?(?:new\s+)?medication\b",
    r"\bstop\s+(?:the\s+)?medication\b",
    r"\bdiscontinue\s+(?:the\s+)?medication\b",
    r"\bchange\s+(?:the\s+)?medication\b",
    r"\badjust\s+(?:the\s+)?(?:medication|dose|dosage)\b",
    r"\bincrease\s+(?:the\s+)?dose\b",
    r"\bdecrease\s+(?:the\s+)?dose\b",
    r"\bchange\s+(?:the\s+)?dose\b",
    r"\brecommend\s+(?:a\s+)?(?:medication|drug)\b",
]

DIAGNOSIS_CLAIM_PATTERNS = [
    r"\byou have\b",
    r"\bthe patient has\b",
    r"\bthis patient has\b",
    r"\bconfirmed diagnosis\b",
    r"\bnew diagnosis\b",
    r"\bdiagnose the patient\b",
]

REQUIRED_REVIEW_PHRASES = [
    "clinician review",
    "qualified clinician",
    "medical review",
    "healthcare professional",
]


In [89]:
def policy_violations(report: AgentReport) -> list[str]:
    text_parts: list[str] = [
        report.summary,
        report.safety_note,
    ]

    for action in report.prioritized_actions:
        text_parts.extend(action.evidence)
        text_parts.extend(action.recommended_follow_up)

    full_text = " ".join(text_parts)

    violations: list[str] = []

    for pattern in BLOCKED_PATTERNS:
        if re.search(pattern, full_text, flags=re.IGNORECASE):
            violations.append(
                f"Medication or prescribing instruction detected: {pattern}"
            )

    for pattern in DIAGNOSIS_CLAIM_PATTERNS:
        if re.search(pattern, full_text, flags=re.IGNORECASE):
            violations.append(
                f"Potential diagnosis claim detected: {pattern}"
            )

    lowered_text = full_text.lower()

    if not any(
        phrase in lowered_text
        for phrase in REQUIRED_REVIEW_PHRASES
    ):
        violations.append(
            "The report does not explicitly require qualified clinician review."
        )

    valid_patient_ids = set(
        df["patient_id"].astype(str).str.upper()
    )

    for action in report.prioritized_actions:
        if action.patient_id.upper() not in valid_patient_ids:
            violations.append(
                f"Unknown patient ID in report: {action.patient_id}"
            )

    return violations

In [90]:

# LangGraph agentic loop
SYSTEM_PROMPT = """
You are MediCare's care-coordination decision-support agent.

Governance rules:

1. Use tools before making patient-specific conclusions.
2. Use only facts returned by tools.
3. Do not invent missing facts.
4. Do not diagnose or make new diagnosis claims.
5. Do not prescribe.
6. Do not recommend starting, stopping, changing, or adjusting medication or dosage.
7. Do not replace emergency services.
8. You may recommend:
   - care-coordination outreach;
   - appointment scheduling;
   - reminder calls;
   - repeat assessment arranged through a clinician;
   - clinician review;
   - escalation to appropriate clinical staff.
9. Never compare raw numeric laboratory values across different tests or units.
10. For missed-appointment requests, call get_missed_appointments first.
11. Use get_patient for complete context when necessary.
12. Every recommendation must require qualified clinician review.

Your final response must be valid JSON only, with exactly this structure:

{
  "summary": "Brief summary",
  "prioritized_actions": [
    {
      "patient_id": "P0001",
      "priority": "High",
      "evidence": [
        "Fact returned by a tool"
      ],
      "recommended_follow_up": [
        "Care-coordination action requiring qualified clinician review"
      ]
    }
  ],
  "safety_note": "All recommendations require qualified clinician review."
}

Do not include Markdown fences.
Do not include patient_name.
"""

In [91]:
agent_llm = llm_model.bind_tools(TOOLS)


In [92]:
class AgentState(TypedDict, total=False):
    messages: Annotated[list[AnyMessage], add_messages]
    turns: int
    audit_passed: bool
    thread_id: str


In [93]:
def call_model(state: AgentState) -> dict[str, Any]:
    thread_id = state.get("thread_id", "unknown")
    next_turn = state.get("turns", 0) + 1

    write_audit_event(
        event_type="model_call",
        thread_id=thread_id,
        details={
            "turn": next_turn,
        },
    )

    response = agent_llm.invoke(
        [
            SystemMessage(content=SYSTEM_PROMPT),
            *state["messages"],
        ]
    )

    return {
        "messages": [response],
        "turns": next_turn,
        "audit_passed": False,
        "thread_id": thread_id,
    }


In [94]:
def execute_tools(state: AgentState) -> dict[str, Any]:
    last_message = state["messages"][-1]
    thread_id = state.get("thread_id", "unknown")
    outputs: list[ToolMessage] = []

    for call in getattr(last_message, "tool_calls", []):
        tool_name = call.get("name")
        tool_args = call.get("args", {})

        write_audit_event(
            event_type="tool_call",
            thread_id=thread_id,
            details={
                "tool": tool_name,
                "argument_keys": list(tool_args.keys()),
            },
        )

        try:
            if tool_name not in TOOL_MAP:
                result = json.dumps({
                    "error": "Requested tool is not allow-listed."
                })

                write_audit_event(
                    event_type="tool_blocked",
                    thread_id=thread_id,
                    details={
                        "tool": tool_name,
                    },
                )
            else:
                result = TOOL_MAP[tool_name].invoke(tool_args)

        except ValidationError:
            result = json.dumps({
                "error": "Tool arguments failed validation."
            })

            write_audit_event(
                event_type="tool_error",
                thread_id=thread_id,
                details={
                    "tool": tool_name,
                    "reason": "argument_validation_failed",
                },
            )

        except Exception:
            # Do not expose raw exception details to the model.
            result = json.dumps({
                "error": "Tool execution failed."
            })

            write_audit_event(
                event_type="tool_error",
                thread_id=thread_id,
                details={
                    "tool": tool_name,
                    "reason": "execution_failed",
                },
            )

        outputs.append(
            ToolMessage(
                content=str(result),
                tool_call_id=call["id"],
            )
        )

    return {
        "messages": outputs,
        "thread_id": thread_id,
    }


In [95]:
def audit_final(state: AgentState) -> dict[str, Any]:
    thread_id = state.get("thread_id", "unknown")
    last_message = state["messages"][-1]
    content = last_message.content

    if not isinstance(content, str):
        content = str(content)

    try:
        report = AgentReport.model_validate_json(content)
    except Exception:
        write_audit_event(
            event_type="blocked_report",
            thread_id=thread_id,
            details={
                "reason": "invalid_json_or_schema",
            },
        )

        retry_message = HumanMessage(
            content=(
                "AUDIT RETRY: Return valid JSON only. Use exactly these top-level "
                "fields: summary, prioritized_actions, and safety_note. "
                "Each action requires patient_id, priority, evidence, and "
                "recommended_follow_up. Explicitly state that qualified clinician "
                "review is required. Do not diagnose, prescribe, or recommend "
                "medication or dose changes. Do not use Markdown fences."
            )
        )

        return {
            "messages": [retry_message],
            "audit_passed": False,
            "thread_id": thread_id,
        }

    violations = policy_violations(report)

    if violations:
        write_audit_event(
            event_type="blocked_report",
            thread_id=thread_id,
            details={
                "reason": "policy_violation",
                "violation_count": len(violations),
                "violations": violations,
            },
        )

        retry_message = HumanMessage(
            content=(
                "AUDIT RETRY: The previous report failed the safety policy. "
                "Return valid JSON only. Do not diagnose, prescribe, or recommend "
                "starting, stopping, changing, or adjusting medications or doses. "
                "Use only facts returned by tools. Every action must be a "
                "care-coordination or clinician-review action. Explicitly require "
                "qualified clinician review.\n\n"
                f"Policy findings: {json.dumps(violations)}"
            )
        )

        return {
            "messages": [retry_message],
            "audit_passed": False,
            "thread_id": thread_id,
        }

    approved_json = report.model_dump_json(indent=2)

    write_audit_event(
        event_type="approved_report",
        thread_id=thread_id,
        details={
            "action_count": len(report.prioritized_actions),
            "priorities": [
                action.priority
                for action in report.prioritized_actions
            ],
        },
    )

    return {
        "messages": [
            HumanMessage(content=approved_json)
        ],
        "audit_passed": True,
        "thread_id": thread_id,
    }


In [96]:
def fallback_node(state: AgentState) -> dict[str, Any]:
    thread_id = state.get("thread_id", "unknown")

    fallback_report = {
        "summary": (
            "The agent could not produce a validated structured report "
            "within the configured turn limit."
        ),
        "prioritized_actions": [],
        "safety_note": (
            "No action should be taken from this output. "
            "Qualified clinician review is required."
        ),
    }

    write_audit_event(
        event_type="fallback_report",
        thread_id=thread_id,
        details={
            "reason": "maximum_turns_exceeded",
        },
    )

    return {
        "messages": [
            HumanMessage(
                content=json.dumps(
                    fallback_report,
                    indent=2,
                )
            )
        ],
        "audit_passed": True,
        "thread_id": thread_id,
    }

In [97]:
MAX_TURNS = 8

In [98]:
def route_after_agent(state: AgentState) -> str:
    last_message = state["messages"][-1]
    tool_calls = getattr(last_message, "tool_calls", [])

    if tool_calls and state.get("turns", 0) < MAX_TURNS:
        return "tools"

    return "audit"


def route_after_audit(state: AgentState) -> str:
    if state.get("audit_passed", False):
        return END

    if state.get("turns", 0) >= MAX_TURNS:
        return "fallback"

    return "agent"

In [99]:
workflow = StateGraph(AgentState)

workflow.add_node("agent", call_model)
workflow.add_node("tools", execute_tools)
workflow.add_node("audit", audit_final)
workflow.add_node("fallback", fallback_node)

In [100]:
workflow.add_edge(START, "agent")

workflow.add_conditional_edges(
    "agent",
    route_after_agent,
    {
        "tools": "tools",
        "audit": "audit",
    },
)

workflow.add_edge("tools", "agent")

workflow.add_conditional_edges(
    "audit",
    route_after_audit,
    {
        "agent": "agent",
        "fallback": "fallback",
        END: END,
    },
)

workflow.add_edge("fallback", END)

In [101]:
memory = MemorySaver()
app = workflow.compile(checkpointer=memory)

In [102]:
# Agent runner


def run_agent(
    request: str,
    thread_id: str | None = None,
) -> dict[str, Any]:
    """
    Run the agent.

    Use a new thread_id for an independent analysis.
    Reuse a thread_id only when conversational memory is intended.
    """
    if not request or not request.strip():
        raise ValueError("request cannot be empty.")

    if thread_id is None:
        thread_id = f"thread_{uuid.uuid4().hex}"

    config = {
        "configurable": {
            "thread_id": thread_id,
        }
    }

    initial_state: AgentState = {
        "messages": [
            HumanMessage(content=request.strip())
        ],
        "turns": 0,
        "audit_passed": False,
        "thread_id": thread_id,
    }

    result = app.invoke(
        initial_state,
        config=config,
    )

    final_message = result["messages"][-1]
    final_text = final_message.content

    if not isinstance(final_text, str):
        final_text = str(final_text)

    try:
        report = AgentReport.model_validate_json(final_text)

        violations = policy_violations(report)

        if violations:
            write_audit_event(
                event_type="runner_rejection",
                thread_id=thread_id,
                details={
                    "violations": violations,
                },
            )

            return {
                "summary": "The final report failed the safety policy.",
                "prioritized_actions": [],
                "safety_note": (
                    "No action should be taken from this output. "
                    "Qualified clinician review is required."
                ),
            }

        return report.model_dump()

    except Exception:
        write_audit_event(
            event_type="runner_parse_failure",
            thread_id=thread_id,
            details={},
        )

        return {
            "summary": (
                "The agent returned an unvalidated response. "
                "No clinical conclusion should be drawn."
            ),
            "prioritized_actions": [],
            "safety_note": (
                "No action should be taken from this output. "
                "Qualified clinician review is required."
            ),
        }


In [103]:
import uuid

In [106]:

# Single-patient analysis
patient_id = "P0003"

single_patient_thread = f"patient_review_{patient_id}_{uuid.uuid4().hex}"

single_patient_report = run_agent(
    request=(
        f"Analyze patient {patient_id}. "
        "First retrieve the full patient profile. "
        "Assess follow-up priority and provide a care-coordination plan "
        "that requires qualified clinician review. "
        "Do not diagnose or recommend medication changes."
    ),
    thread_id=single_patient_thread,
)

print("\n--- Task 4: Single Patient Analysis ---")
print(json.dumps(single_patient_report, indent=2))


--- Task 4: Single Patient Analysis ---
{
  "summary": "Patient P0003 is a 45-year-old male with Type 2 Diabetes and CKD. Recent labs show an elevated HbA1c of 11.4%. He is on insulin glargine and losartan. His last visit was 149 days ago, and his next visit is scheduled. SpO2 is 91%. Medication dose was adjusted at the last visit.",
  "prioritized_actions": [
    {
      "patient_id": "P0003",
      "priority": "High",
      "evidence": [
        "HbA1c is 11.4%, indicating poor glycemic control.",
        "SpO2 is 91%, which is below normal range.",
        "Medication dose was adjusted at last visit.",
        "Patient has comorbid CKD and diabetes."
      ],
      "recommended_follow_up": [
        "Care-coordination outreach to assess current symptoms and adherence.",
        "Arrange clinician review for recent medication adjustment and persistently elevated HbA1c.",
        "Schedule repeat assessment of glycemic control and oxygen saturation through a clinician.",
        "Esc

In [107]:
#  MemorySaver follow-up

follow_up_report = run_agent(
    request=(
        f"For patient {patient_id}, summarize only the evidence in the "
        "retrieved record that supports the earlier priority. "
        "Return the required JSON format. "
        "Do not add new clinical conclusions."
    ),
    thread_id=single_patient_thread,
)

print("\n--- MemorySaver Follow-Up ---")
print(json.dumps(follow_up_report, indent=2))



--- MemorySaver Follow-Up ---
{
  "summary": "Evidence supporting high follow-up priority for patient P0003 is summarized below.",
  "prioritized_actions": [
    {
      "patient_id": "P0003",
      "priority": "High",
      "evidence": [
        "HbA1c is 11.4%.",
        "SpO2 is 91%.",
        "Medication dose was adjusted at last visit.",
        "Patient has comorbid CKD and diabetes."
      ],
      "recommended_follow_up": [
        "Care-coordination action requiring qualified clinician review"
      ]
    }
  ],
  "safety_note": "All recommendations require qualified clinician review."
}


In [108]:
#-appointment follow-up

missed_thread = f"missed_appointment_follow_up_{uuid.uuid4().hex}"

missed_appointment_report = run_agent(
    request=(
        "Retrieve all patients who missed their last appointment. "
        "Prioritize follow-up using the returned clinical context. "
        "Retrieve individual full profiles when more context is needed. "
        "Provide care-coordination outreach actions for each returned patient. "
        "Do not diagnose, prescribe, or recommend medication changes. "
        "Every action must require qualified clinician review."
    ),
    thread_id=missed_thread,
)

print("\n--- Task 5: Missed Appointment Follow-Up ---")
print(json.dumps(missed_appointment_report, indent=2))



--- Task 5: Missed Appointment Follow-Up ---
{
  "summary": "The agent could not produce a validated structured report within the configured turn limit.",
  "prioritized_actions": [],
  "safety_note": "No action should be taken from this output. Qualified clinician review is required."
}


In [105]:
# inspect metadata-only audit events

print("\n--- Audit Log ---")
print(json.dumps(get_audit_log(), indent=2, default=str))


--- MemorySaver Follow-Up ---
{
  "summary": "Evidence supporting high follow-up priority for patient P0003 is listed below. Qualified clinician review is required for all actions.",
  "prioritized_actions": [
    {
      "patient_id": "P0003",
      "priority": "High",
      "evidence": [
        "HbA1c is 11.4%",
        "SpO2 is 91%",
        "Diagnosis: Type 2 Diabetes, CKD",
        "Medication dose adjusted this visit"
      ],
      "recommended_follow_up": [
        "All actions require qualified clinician review."
      ]
    }
  ],
  "safety_note": "All recommendations require qualified clinician review."
}

--- Task 5: Missed Appointment Follow-Up ---
{
  "summary": "Fifteen patients missed their last appointment, several with complex chronic conditions and concerning recent clinical findings. Highest priority is given to those with recent symptoms, abnormal vital signs, or missed medication doses. All require care-coordination outreach and qualified clinician review.",
  "

In [110]:
# !pip install pytest


In [111]:
def test_patient_id_validation():
    valid = PatientLookup(patient_id="p0001")
    assert valid.patient_id == "P0001"


def test_invalid_patient_id():
    import pytest

    with pytest.raises(Exception):
        PatientLookup(patient_id="invalid-id")


def test_patient_name_is_not_model_facing():
    payload = json.loads(get_patient.invoke({"patient_id": "P0001"}))

    assert len(payload) == 1
    assert "patient_name" not in payload[0]


def test_priority_action_requires_evidence():
    import pytest

    with pytest.raises(Exception):
        PriorityAction(
            patient_id="P0001",
            priority="High",
            evidence=[],
            recommended_follow_up=[
                "Coordinate clinician review."
            ],
        )


In [113]:
def test_policy_blocks_medication_change():
    report = AgentReport(
        summary="Follow-up is needed.",
        prioritized_actions=[
            PriorityAction(
                patient_id="P0001",
                priority="High",
                evidence=["The patient missed an appointment."],
                recommended_follow_up=[
                    "Change the medication dose immediately."
                ],
            )
        ],
        safety_note="Qualified clinician review is required.",
    )

    violations = policy_violations(report)

    assert violations
